# From an aluminum transmon specification to expected measurements

Run from the repository root or set the working directory in the next cell. This notebook uses the tested core scientific route. The capacitance data is an uncalibrated surrogate; it is not a Palace result. See `docs/fem.md` for the full reference chain.

In [ ]:
from pathlib import Path
import os
if not Path("configs/demo.toml").exists():
    os.chdir("..")  # when Jupyter starts inside examples/
from sqw import config, circuit, geometry, cad, experiment
from sqw.io import write_json
cfg = config.load("configs/demo.toml")
out = Path("build/notebook")
out.mkdir(parents=True, exist_ok=True)

## Electrical target

Infer EJ/EC from f01 and anharmonicity using the exact charge-basis spectrum. This target does not overwrite the forward prediction.

In [ ]:
target = circuit.synthesize(cfg)
target

## Shared geometry and mask

The same conductor ownership feeds the GDS and the FEM mesher. The generic junction is a location marker; it needs a qualified process cell.

In [ ]:
geo = geometry.build(cfg)
geometry.save(geo, out / "geometry.json")
cad.write_gds(geo, out / "device.gds")
geometry.preview(geo, out / "layout.png")
from IPython.display import display, Image
display(Image(filename=str(out / "layout.png")))

## Forward quantum model

The floating-pad common mode is eliminated explicitly. A half-wave antinode approximation maps static readout capacitance into a modal capacitance.

In [ ]:
extraction = circuit.surrogate(cfg, geo)
params = circuit.parameters(cfg, extraction)
write_json(out / "parameters.json", params)
{k: params[k] for k in ["source", "f01_GHz", "anharmonicity_MHz", "g_MHz", "chi_exact_MHz", "target_deviation"]}

## Expected experiment

Intrinsic T1, Tphi and readout Qi/Qc are input assumptions. The simulator adds a single-mode Purcell estimate and seeded shot/assignment noise.

In [ ]:
metrics = experiment.simulate(cfg, params, out / "experiment")
display(Image(filename=str(out / "experiment/measurements.png")))
metrics

## Explore a change

Increase pad width, rebuild, and compare f01. For real design iterations replace the surrogate with converged Palace output and validate against the measured reference chip.

In [ ]:
import copy
changed = copy.deepcopy(cfg)
changed["layout"]["pad_width_um"] += 20
geo2 = geometry.build(changed)
p2 = circuit.parameters(changed, circuit.surrogate(changed, geo2))
print(f"f01: {params["f01_GHz"]:.6f} -> {p2["f01_GHz"]:.6f} GHz")